TODO:
- pegar a parte do dataset que ta cropada
	- listar por idades
	- listar por generos
	- listar por generos e idades
	- listar por racas (vai ser importante quando começar a dar erro com gente branca, por exemplo) 

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jangedoo/utkface-new")

print("Path to dataset files:", path)

Path to dataset files: /home/pedroln/.cache/kagglehub/datasets/jangedoo/utkface-new/versions/1


In [20]:
import pandas as pd
import os

data = []
path = "../data/1/utkface_aligned_cropped/UTKFace/"

for file in os.listdir(path):
    if not file.endswith(".jpg"):
        continue
    
    parts = file.split("_")

    if len(parts) < 3:
        continue
    
    try:
        idade = int(parts[0])
        genero = int(parts[1])
        raca = int(parts[2])
        
        data.append({
            "idade": idade,
            "genero": genero,
            "raca": raca,
            "arquivo": file
        })
        
    except:
        continue

df = pd.DataFrame(data)

### Legendas

As imagens são legendadas no formato `[idade]_[genero]_[raça]_[data&hora].jpg`, seguindo a lógica:
- [idade] é um inteiro de 0 a 116, que indica a idade do indivíduo na foto.
- [genero] é um inteiro que descreve o genero do indivíduo na foto, sendo:
  - 0 - homem
  - 1 - mulher
- [raça] é um inteiro de 0 a 4, denotando a etnia do indivíduo na foto, sendo:
  - 0 - Branco
  - 1 - Preto
  - 2 - Asiático
  - 3 - Indiano
  - 4 - Outros
- [data&hora] estão no formato yyyymmddHHMMSSFFF, indicando a data e hora que a foto foi inserida no dataset da UTKFace.


In [28]:
bins = [0, 5, 12, 19, 29, 59, 120]
labels = ["Criança Pequena", "Criança", "Adolescente", "Jovem adulto", "Adulto", "Idoso"]

df["faixa_etaria"] = pd.cut(
    df["idade"],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True
)

df["faixa_etaria"].value_counts().sort_index()

faixa_etaria
Criança Pequena    2363
Criança            1050
Adolescente        1180
Jovem adulto       7344
Adulto             9080
Idoso              2688
Name: count, dtype: int64

In [36]:
df["genero_label"] = df["genero"].map({0: "Masculino", 1: "Feminino"})
df["genero_label"].value_counts()

genero_label
Masculino    12391
Feminino     11314
Name: count, dtype: int64

In [37]:
df.groupby("genero_label")["idade"].agg(["mean", "median", "min", "max"])
pd.crosstab(df["faixa_etaria"], df["genero_label"])

genero_label,Feminino,Masculino
faixa_etaria,,
Criança Pequena,1207,1156
Criança,599,451
Adolescente,659,521
Jovem adulto,4282,3062
Adulto,3425,5655
Idoso,1142,1546


In [38]:
raca_map = {
    0: "Branco",
    1: "Negro",
    2: "Asiático",
    3: "Indiano",
    4: "Outro"
}

df["raca_label"] = df["raca"].map(raca_map)

df["raca_label"].value_counts()

pd.crosstab(df["raca_label"], df["genero_label"])


genero_label,Feminino,Masculino
raca_label,,
Asiático,1859,1575
Branco,4601,5477
Indiano,1714,2261
Negro,2208,2318
Outro,932,760
